# Ch04 练习参考答案：MC vs TD(0) 在 GridWorld

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'rlenvs').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from utils import set_seed
from rlenvs import small_grid_5x5

set_seed(0)


def td0_estimate(env, gamma=0.9, alpha=0.05, n_episodes=2000):
    V = np.zeros(env.nS)
    rms_history = []
    for ep in range(n_episodes):
        s = env.reset()
        done = False
        while not done:
            a = np.random.randint(env.nA)
            s_next, r, done, _ = env.step(a)
            td_target = r + (0 if done else gamma * V[s_next])
            V[s] += alpha * (td_target - V[s])
            s = s_next
        rms_history.append(np.sqrt(np.mean(V ** 2)))
    return V, np.array(rms_history)


def mc_first_visit(env, gamma=0.9, alpha=0.01, n_episodes=2000):
    V = np.zeros(env.nS)
    rms_history = []
    for ep in range(n_episodes):
        s = env.reset()
        traj = []
        done = False
        while not done:
            a = np.random.randint(env.nA)
            s_next, r, done, _ = env.step(a)
            traj.append((s, r))
            s = s_next
        G = 0
        Gs = []
        for s, r in reversed(traj):
            G = r + gamma * G
            Gs.append((s, G))
        seen = set()
        for s, G in reversed(Gs):
            if s in seen:
                continue
            seen.add(s)
            V[s] += alpha * (G - V[s])
        rms_history.append(np.sqrt(np.mean(V ** 2)))
    return V, np.array(rms_history)


env = small_grid_5x5(seed=0)
n_seeds = 30
n_eps = 1500
td_runs = np.zeros((n_seeds, n_eps))
mc_runs = np.zeros((n_seeds, n_eps))
for seed in range(n_seeds):
    np.random.seed(seed)
    env = small_grid_5x5(seed=seed)
    _, td_hist = td0_estimate(env, alpha=0.05, n_episodes=n_eps)
    _, mc_hist = mc_first_visit(env, alpha=0.01, n_episodes=n_eps)
    td_runs[seed] = td_hist
    mc_runs[seed] = mc_hist

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(td_runs.mean(0), label='TD(0) α=0.05', linewidth=2)
ax.plot(mc_runs.mean(0), label='MC α=0.01', linewidth=2)
ax.set_xlabel('episode')
ax.set_ylabel('RMS(V) over runs')
ax.set_title('TD vs MC：TD 早期下降快，MC 后期更准')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("观察：")
print("- TD 早期下降快（bootstrap 让信号快速传播）")
print("- MC 后期更接近真值（无偏）")
print("- 这正是 Ch04 讲的 bias-variance tradeoff")